# MNIST con PyTorch. Red Neuronal Feedforward

Implementación de una red neuronal feedforward para clasificación multiclase del dataset MNIST usando PyTorch.

## 1. Carga del dataset MNIST

In [10]:
import torch
from torch.utils.data import random_split
from torchvision import datasets, transforms
import torch.nn as nn
import torch.optim as optim

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Usando: {device}")

# Transformación: convierte imágenes PIL a tensores y normaliza píxeles a [0, 1]
transform = transforms.ToTensor()

# MNIST viene con 60,000 para train y 10,000 para test
full_train_dataset = datasets.MNIST(root='../data', train=True,  download=True, transform=transform)
test_dataset       = datasets.MNIST(root='../data', train=False, download=True, transform=transform)

# 10,000 muestras del train para validación
train_dataset, val_dataset = random_split(full_train_dataset, [50_000, 10_000])

print(f"Train:      {len(train_dataset):,} muestras")
print(f"Validación: {len(val_dataset):,} muestras")
print(f"Test:       {len(test_dataset):,} muestras")

Usando: cuda
Train:      50,000 muestras
Validación: 10,000 muestras
Test:       10,000 muestras


## 2. Definición de la red neuronal

In [11]:
class FeedForwardNet(nn.Module):
    """    
    Como parámetro se recibe una lista con el número de neuronas por capa oculta

    Arquitectura: Entrada (784) → [Linear → ReLU] x n_capas → Output (10)
    """
    def __init__(self, hidden_sizes: list[int]):
        super().__init__()
        
        layer_sizes = [784] + hidden_sizes + [10]
        
        layers = []
        for i in range(len(layer_sizes) - 1):
            # Añadir capas lineales y ReLU
            layers.append(nn.Linear(layer_sizes[i], layer_sizes[i + 1])) 
            if i < len(layer_sizes) - 2:   # No ReLU en la última capa
                layers.append(nn.ReLU())
        
        self.network = nn.Sequential(*layers)
    
    def forward(self, x):
        x = x.view(x.size(0), -1)   # Aplana 28x28 a 784
        return self.network(x)


# Ver que todo está bien con la arquitectura
model_test = FeedForwardNet(hidden_sizes=[256, 128])
print(model_test)

FeedForwardNet(
  (network): Sequential(
    (0): Linear(in_features=784, out_features=256, bias=True)
    (1): ReLU()
    (2): Linear(in_features=256, out_features=128, bias=True)
    (3): ReLU()
    (4): Linear(in_features=128, out_features=10, bias=True)
  )
)


## 3. Función de pérdida y optimizador

Aquí usamos Cross Entropy como función de pérdida porque es un problema de clasificación multiclase. Para el optimizador, una función en el que puedes elegir entre adam o sgd. 

In [12]:
# Función de pérdida
criterion = nn.CrossEntropyLoss()

# Optimizadores 
def make_optimizer(model, optimizer_name='adam', lr=1e-3):
    if optimizer_name == 'adam':
        return optim.Adam(model.parameters(), lr=lr)
    elif optimizer_name == 'sgd':
        return optim.SGD(model.parameters(), lr=lr)
    else:
        raise ValueError(f"Optimizador desconocido: {optimizer_name}")

## 4. Loop de entrenamiento

In [13]:
from torch.utils.data import DataLoader
from tqdm import tqdm

def train(model, optimizer, epochs, batch_size=128):
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    val_loader   = DataLoader(val_dataset,   batch_size=batch_size, shuffle=False)
    
    history = {'train_loss': [], 'val_loss': [], 'val_acc': []}
    
    for epoch in range(1, epochs + 1):
        # Fase de entrenamiento 
        model.train()
        running_loss = 0.0
        
        loop = tqdm(train_loader, desc=f"Época {epoch}/{epochs}", leave=False)
        for X_batch, y_batch in loop:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            
            optimizer.zero_grad()
            logits = model(X_batch)
            loss = criterion(logits, y_batch)
            loss.backward()
            optimizer.step()
            
            running_loss += loss.item()
            loop.set_postfix(loss=f"{loss.item():.4f}")
        
        train_loss = running_loss / len(train_loader)
        
        # Fase de validación
        model.eval()
        val_loss = 0.0
        correct  = 0
        
        with torch.no_grad():
            for X_batch, y_batch in val_loader:
                X_batch, y_batch = X_batch.to(device), y_batch.to(device)
                
                logits = model(X_batch)
                val_loss += criterion(logits, y_batch).item()
                preds = logits.argmax(dim=1)
                correct += (preds == y_batch).sum().item()
        
        val_loss /= len(val_loader)
        val_acc   = correct / len(val_dataset)
        
        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        history['val_acc'].append(val_acc)
        
        print(f"Época {epoch:>2}/{epochs} | "
              f"Train loss: {train_loss:.4f} | "
              f"Val loss: {val_loss:.4f} | "
              f"Val acc: {val_acc:.4f}")
    
    return history

## 5. Entrenamiento de modelos

Se entrenan 3 redes variando arquitectura, optimizador, batch size, épocas y learning rate.

| Modelo | Capas ocultas | Optimizador | Batch size | Épocas | LR |
|--------|--------------|-------------|-----------|--------|-----|
| 1 | [128] | Adam | 128 | 10 | 1e-3 |
| 2 | [256, 128] | Adam | 256 | 15 | 1e-3 |
| 3 | [512, 256, 128] | SGD | 64 | 20 | 1e-2 |

### Modelo 1 — Red pequeña, Adam, batch 128, 10 épocas

In [16]:
model1 = FeedForwardNet(hidden_sizes=[128]).to(device)
optimizer1 = make_optimizer(model1, 'adam', lr=1e-3)

print("Arquitectura:")
print(model1)

history1 = train(model1, optimizer1, epochs=10, batch_size=128)

Arquitectura:
FeedForwardNet(
  (network): Sequential(
    (0): Linear(in_features=784, out_features=128, bias=True)
    (1): ReLU()
    (2): Linear(in_features=128, out_features=10, bias=True)
  )
)


Época  1/10 | Train loss: 0.4502 | Val loss: 0.2595 | Val acc: 0.9247


Época  2/10 | Train loss: 0.2199 | Val loss: 0.1948 | Val acc: 0.9408


Época  3/10 | Train loss: 0.1640 | Val loss: 0.1628 | Val acc: 0.9523


Época  4/10 | Train loss: 0.1285 | Val loss: 0.1334 | Val acc: 0.9596


...

Época  6/10 | Train loss: 0.0871 | Val loss: 0.1094 | Val acc: 0.9669


Época  7/10 | Train loss: 0.0726 | Val loss: 0.1068 | Val acc: 0.9664


Época  8/10 | Train loss: 0.0612 | Val loss: 0.1032 | Val acc: 0.9697


Época  9/10 | Train loss: 0.0532 | Val loss: 0.0917 | Val acc: 0.9722


Época 10/10 | Train loss: 0.0450 | Val loss: 0.0924 | Val acc: 0.9696


### Modelo 2 — Red mediana, Adam, batch 256, 15 épocas

In [ ]:
model2 = FeedForwardNet(hidden_sizes=[256, 128]).to(device)
optimizer2 = make_optimizer(model2, 'adam', lr=1e-3)

print("Arquitectura:")
print(model2)

history2 = train(model2, optimizer2, epochs=15, batch_size=256)

Arquitectura:
FeedForwardNet(
  (network): Sequential(
    (0): Linear(in_features=784, out_features=256, bias=True)
    (1): ReLU()
    (2): Linear(in_features=256, out_features=128, bias=True)
    (3): ReLU()
    (4): Linear(in_features=128, out_features=10, bias=True)
  )
)
Parámetros: 235,146



Época  1/15 | Train loss: 0.4833 | Val loss: 0.2334 | Val acc: 0.9312


Época  2/15 | Train loss: 0.1887 | Val loss: 0.1696 | Val acc: 0.9496


Época  3/15 | Train loss: 0.1336 | Val loss: 0.1343 | Val acc: 0.9617


Época  4/15 | Train loss: 0.1004 | Val loss: 0.1083 | Val acc: 0.9667


...

Época 11/15 | Train loss: 0.0223 | Val loss: 0.0899 | Val acc: 0.9761


Época 12/15 | Train loss: 0.0179 | Val loss: 0.0912 | Val acc: 0.9758


Época 13/15 | Train loss: 0.0154 | Val loss: 0.0849 | Val acc: 0.9784


Época 14/15 | Train loss: 0.0117 | Val loss: 0.0848 | Val acc: 0.9785


Época 15/15 | Train loss: 0.0077 | Val loss: 0.0881 | Val acc: 0.9783


### Modelo 3 — Red profunda, SGD, batch 64, 20 épocas

In [17]:
model3 = FeedForwardNet(hidden_sizes=[512, 256, 128]).to(device)
optimizer3 = make_optimizer(model3, 'sgd', lr=1e-2)

print("Arquitectura:")
print(model3)

history3 = train(model3, optimizer3, epochs=20, batch_size=64)

Arquitectura:
FeedForwardNet(
  (network): Sequential(
    (0): Linear(in_features=784, out_features=512, bias=True)
    (1): ReLU()
    (2): Linear(in_features=512, out_features=256, bias=True)
    (3): ReLU()
    (4): Linear(in_features=256, out_features=128, bias=True)
    (5): ReLU()
    (6): Linear(in_features=128, out_features=10, bias=True)
  )
)


Época  1/20 | Train loss: 2.2268 | Val loss: 1.9864 | Val acc: 0.5513


Época  2/20 | Train loss: 1.1313 | Val loss: 0.6399 | Val acc: 0.8175


Época  3/20 | Train loss: 0.5181 | Val loss: 0.4308 | Val acc: 0.8802


Época  4/20 | Train loss: 0.3973 | Val loss: 0.3733 | Val acc: 0.8940


...

Época 16/20 | Train loss: 0.1410 | Val loss: 0.1652 | Val acc: 0.9512


Época 17/20 | Train loss: 0.1324 | Val loss: 0.1619 | Val acc: 0.9523


Época 18/20 | Train loss: 0.1242 | Val loss: 0.1446 | Val acc: 0.9579


Época 19/20 | Train loss: 0.1171 | Val loss: 0.1627 | Val acc: 0.9534


Época 20/20 | Train loss: 0.1097 | Val loss: 0.1368 | Val acc: 0.9597


### Comparación de modelos

Primero comparamos usando el validation para elegir el mejor

In [20]:
resultados = [
    {"nombre": "Modelo 1", "modelo": model1, "history": history1, "config": "[128] | Adam | batch=128 | lr=1e-3 | 10 épocas"},
    {"nombre": "Modelo 2", "modelo": model2, "history": history2, "config": "[256,128] | Adam | batch=256 | lr=1e-3 | 15 épocas"},
    {"nombre": "Modelo 3", "modelo": model3, "history": history3, "config": "[512,256,128] | SGD | batch=64 | lr=1e-2 | 20 épocas"},
]

print(f"{'Modelo':<10} {'Val Loss':>10} {'Val Acc':>10}  Configuración")
print("-" * 75)
for r in resultados:
    best_acc  = max(r["history"]["val_acc"])
    best_loss = min(r["history"]["val_loss"])
    print(f"{r['nombre']:<10} {best_loss:>10.4f} {best_acc:>10.4f}  {r['config']}")

mejor = max(resultados, key=lambda r: max(r["history"]["val_acc"]))
print(f"\nMejor modelo: {mejor['nombre']} ({mejor['config']})")

Modelo       Val Loss    Val Acc  Configuración
---------------------------------------------------------------------------
Modelo 1       0.0917     0.9722  [128] | Adam | batch=128 | lr=1e-3 | 10 épocas
Modelo 2       0.0820     0.9785  [256,128] | Adam | batch=256 | lr=1e-3 | 15 épocas
Modelo 3       0.1368     0.9597  [512,256,128] | SGD | batch=64 | lr=1e-2 | 20 épocas

Mejor modelo: Modelo 2 ([256,128] | Adam | batch=256 | lr=1e-3 | 15 épocas)


## 6. Evaluación final en test

In [19]:
test_loader = DataLoader(test_dataset, batch_size=256, shuffle=False)

best_model = mejor["modelo"]
best_model.eval()

test_loss = 0.0
correct   = 0

with torch.no_grad():
    for X_batch, y_batch in test_loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        logits = best_model(X_batch)
        test_loss += criterion(logits, y_batch).item()
        correct += (logits.argmax(dim=1) == y_batch).sum().item()

test_loss /= len(test_loader)
test_acc   = correct / len(test_dataset)

print(f"Modelo evaluado: {mejor['nombre']} — {mejor['config']}")
print(f"Test loss:       {test_loss:.4f}")
print(f"Test accuracy:   {test_acc:.4f}")

Modelo evaluado: Modelo 2 — [256,128] | Adam | batch=256 | lr=1e-3 | 15 épocas
Test loss:       0.0788
Test accuracy:   0.9790
